# Inimene-tsüklis tööprotsess Microsoft Agent Frameworkiga

## 🎯 Õpieesmärgid

Selles märkmes õpid, kuidas rakendada **inimene-tsüklis** tööprotsesse, kasutades Microsoft Agent Frameworki `RequestInfoExecutor`-i. See võimas muster võimaldab AI töövooge peatada, et koguda inimeste sisendit, muutes oma agendid interaktiivseks ja andes inimestele kontrolli oluliste otsuste üle.

## 🔄 Mis on Inimene-tsüklis?

**Inimene-tsüklis (HITL)** on disainimuster, kus tehisintellekti agendid peatavad täitmise, et paluda inimeste sisendit enne jätkamist. See on oluline:

- ✅ **Olulised otsused** - Saada inimeste heakskiit enne tähtsate toimingute tegemist
- ✅ **Ebamäärased olukorrad** - Lase inimestel selgitada, kui AI on ebakindel
- ✅ **Kasutajasoovid** - Küsi kasutajatelt valida mitme võimaluse vahel
- ✅ **Vastavus ja ohutus** - Tagada inimeste järelevalve reguleeritud tegevuste puhul
- ✅ **Interaktiivsed kogemused** - Loo vestlusagent, mis reageerib kasutaja sisendile

## 🏗️ Kuidas see töötab Microsoft Agent Frameworkis

Raamistik pakub HITL jaoks kolme põhikomponenti:

1. **`RequestInfoExecutor`** - eriline täitja, mis peatab töövoo ja käivitab `RequestInfoEvent` ürituse
2. **`RequestInfoMessage`** - baasklass tüübistatud taotlusandmete jaoks, mis saadetakse inimestele
3. **`RequestResponse`** - seob inimeste vastused algsete taotlustega `request_id` abil

**Töövoo muster:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 Meie näide: Hotelli broneerimine koos kasutaja kinnitusega

Täiendame tingimuslikku töövoogu, lisades inimkinnituse **enne** alternatiivsete sihtkohtade pakkumist:

1. Kasutaja küsib sihtkohta (nt "Pariis")
2. `availability_agent` kontrollib, kas tube on saadaval
3. **Kui tube pole** → `confirmation_agent` küsib "Kas soovite näha alternatiive?"
4. Töövoog **peatub** `RequestInfoExecutor` abil
5. **Inimene vastab** "jah" või "ei" konsooli sisendi kaudu
6. `decision_manager` juhib vastavalt vastusele:
   - **Jah** → Näita alternatiivseid sihtkohti
   - **Ei** → Tühista broneeringu päring
7. Kuvatakse lõpptulemus

See näitab, kuidas anda kasutajatele kontroll agentide soovituste üle!

---

Alustame! 🚀


## 1. samm: vajalike teekide importimine

Impordime standardse Agendi raamistikukomponendid pluss **inimene-tsüklis spetsiifilised klassid**:
- `RequestInfoExecutor` – täitja, kes peatab töövoo inimese sisendi ootamiseks
- `RequestInfoEvent` – sündmus, mis käivitub inimese sisendi taotlemisel
- `RequestInfoMessage` – tüübistatud päringu alusklass
- `RequestResponse` – seob inimese vastused päringutega
- `WorkflowOutputEvent` – sündmus töövoo väljundite tuvastamiseks


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    ChatMessage,
    Executor,
    RequestInfoEvent,          # NEW: Event when human input is requested
    RequestInfoExecutor,       # NEW: Executor that gathers human input
    RequestInfoMessage,        # NEW: Base class for request payloads
    RequestResponse,           # NEW: Correlates response with request
    Role,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowOutputEvent,       # NEW: Event for workflow outputs
    WorkflowRunState,          # NEW: Enum of workflow run states
    WorkflowStatusEvent,       # NEW: Event for run state changes
    ai_function,
    executor,
    handler,                   # NEW: Decorator for executor methods
)

# 🤖 Azure OpenAI client integration
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## Samm 2: Määra Pydantic mudelid struktureeritud väljundite jaoks

Need mudelid määratlevad **skeemi**, mida agendid tagastavad. Säilitame kõik mudelid tingimuslikust töövoost ja lisame:

**Uus inimsekkumise jaoks:**
- `HumanFeedbackRequest` - `RequestInfoMessage` alamklass, mis määratleb inimesele saadetava päringu andmepaketi
  - Sisaldab `prompt` (küsimus, mida esitada) ja `destination` (kontekst kättesaamatu linna kohta)


In [22]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Dataclass for RequestInfoExecutor
@dataclass
class HumanFeedbackRequest(RequestInfoMessage):
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## Samm 3: Loo hotelli broneerimise tööriist

Sama tööriist nagu tingimuslikus töös – kontrollib, kas sihtkohas on tubasid saadaval.


In [23]:
@ai_function(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @ai_function decorator")

✅ hotel_booking tool created with @ai_function decorator


## 4. samm: marsruutimise seisundfunktsioonide määratlemine

Me vajame meie inimsekkumise töövoo jaoks **neli seisundfunktsiooni**:

**Tingimusliku töövoo puhul:**
1. `has_availability_condition` - Marsruudib, kui hotellid ON saadaval
2. `no_availability_condition` - Marsruudib, kui hotellid EI OLE saadaval

**Uus inimsekkumise jaoks:**
3. `user_wants_alternatives_condition` - Marsruudib, kui kasutaja ütleb "jah" alternatiividele
4. `user_declines_alternatives_condition` - Marsruudib, kui kasutaja ütleb "ei" alternatiividele


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## Samm 5: Loo otsusehaldurite täideviija

See on **inimene-silmuses-mustri tuum**! `DecisionManager` on kohandatud `Executor`, mis:

1. **Võtab vastu inimtagasisidet** `RequestResponse` objektide kaudu
2. **Töötleb kasutaja otsust** (jah/ei)
3. **Suunab töösuunda** saates sõnumeid sobivatele agentidele

Peamised omadused:
- Kasutab meetodite eksponeerimiseks töövoo sammudena `@handler` dekoratsiooni
- Võtab vastu `RequestResponse[HumanFeedbackRequest, str]`, mis sisaldab nii algset päringut kui kasutaja vastust
- Edastab lihtsaid "jah" või "ei" sõnumeid, mis käivitavad meie tingimusfunktsioonid


In [25]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_human_feedback(
        self,
        feedback: RequestResponse[HumanFeedbackRequest, str],
        ctx: WorkflowContext[AgentExecutorRequest],
    ) -> None:
        """
        Process human feedback and let the workflow route based on conditions.
        
        The RequestResponse contains:
        - feedback.data: The user's string reply (e.g., "yes" or "no")
        - feedback.original_request: The HumanFeedbackRequest with context
        
        This handler just displays feedback and passes the RequestResponse through.
        The routing is done by condition functions on the edges.
        """
        user_reply = (feedback.data or "").strip().lower()
        destination = getattr(feedback.original_request, "destination", "unknown")

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = ChatMessage(
                Role.USER,
                text=f"The user wants to see alternative destinations near {destination}. Please suggest one.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → Will route to cancellation_agent
                </div>
            """)
            )
            # Create and send a message for the cancellation_agent
            user_msg = ChatMessage(
                Role.USER,
                text="The user has declined to see alternatives. Please acknowledge their decision.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = ChatMessage(
                Role.USER,
                text="The user has declined to see alternatives. Please acknowledge their decision.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created with @handler method for human feedback")

✅ DecisionManager executor created with @handler method for human feedback


## 6. samm: Loo kohandatud kuvamise täitja

Sama kuvamise täitja nagu tingimuslikus töövoos – annab lõplikud tulemused töövoo väljundina.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## Samm 7: Laadi keskkonnamuutujad

Konfigureeri LLM klient (Microsoft Foundry / Azure OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

from azure.identity import AzureCliCredential

# Azure OpenAI via the Responses API. Sign in with `az login` for keyless Entra ID auth.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API,
# so this sample calls Azure OpenAI directly. OpenAIChatClient uses the Responses API.
chat_client = OpenAIChatClient(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    credential=AzureCliCredential(),
    model_id=os.environ["AZURE_OPENAI_DEPLOYMENT"],
)

print("✅ Chat client configured with Azure OpenAI (Responses API)")


✅ Chat client configured with GitHub Models


## Samm 8: Loo tehisintellekti agendid ja täitjad

Loome **kuus töövoo komponenti**:

**Agendid (mähitud AgentExecutor-sse):**
1. **availability_agent** - Kontrollib hotelli saadavust tööriista abil
2. **confirmation_agent** - 🆕 Valmistab ette inimkinnituse taotluse
3. **alternative_agent** - Soovitab alternatiivseid linnu (kui kasutaja ütleb jah)
4. **booking_agent** - Julgustab broneerima (kui tuba on saadaval)
5. **cancellation_agent** - 🆕 Käsitleb tühistamisteadet (kui kasutaja ütleb ei)

**Eritäitjad:**
6. **request_info_executor** - 🆕 `RequestInfoExecutor`, mis peatab töövoo inimese sisendi ootamiseks
7. **decision_manager** - 🆕 Kohandatud täitja, mis suunab vastavalt inimese reaktsioonile (juba ülal määratletud)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        response_format=BookingCheckResult,
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        response_format=ConfirmationQuestion,  # Use Pydantic model for agent output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        response_format=AlternativeResult,
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        response_format=BookingConfirmation,
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        response_format=CancellationMessage,
    ),
    id="cancellation_agent",
)

# NEW: RequestInfoExecutor - pauses workflow to gather human input
request_info_executor = RequestInfoExecutor(id="request_info")

# NEW: DecisionManager instance - routes based on human feedback
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## Samm 9: Töötada välja inimtõlgendusega töövoog

Nüüd koostame töövoo graafi koos **tingimusliku marsruutimisega**, mis sisaldab ka inimtõlgenduse teed:

**Töövoo struktuur:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**Põhivõrgustiku servad:**
- `availability_agent → confirmation_agent` (kui tube pole)
- `confirmation_agent → prepare_human_request` (muudetakse tüüp)
- `prepare_human_request → request_info_executor` (peatub inimese jaoks)
- `request_info_executor → decision_manager` (alati - tagastab RequestResponse)
- `decision_manager → alternative_agent` (kui kasutaja ütleb "jah")
- `decision_manager → cancellation_agent` (kui kasutaja ütleb "ei")
- `availability_agent → booking_agent` (kui tube on saadaval)
- Kõik teed lõpevad `display_result` juures


In [29]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder()
    .set_start_executor(availability_agent)
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, prepare_human_request)  # Transform to HumanFeedbackRequest
    .add_edge(prepare_human_request, request_info_executor)  # Send to RequestInfoExecutor
    .add_edge(request_info_executor, decision_manager)    # Always goes to decision manager
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## Samm 10: Käivita testjuhtum 1 - Linn ILMA saadavuseta (Pariis inimkinnitusega)

See test demonstreerib **täielikku inimese kaasamise tsüklit**:

1. Hotelli päring Pariisis
2. availability_agent kontrollib → Tube pole
3. confirmation_agent loob inimesele suunatud küsimuse
4. request_info_executor **peatab töövoo** ja saadab `RequestInfoEvent`
5. **Rakendus tuvastab sündmuse ja kuvab kasutajale konsoolis juhise**
6. Kasutaja sisestab "jah" või "ei"
7. Rakendus saadab vastuse `send_responses_streaming()` abil
8. decision_manager suunab vastuse põhjal
9. Kuvatakse lõplik tulemus

**Põhimuster:**
- Kasuta esimesel iteratsioonil `workflow.run_stream()`
- Kasuta järgnevatel iteratsioonidel `workflow.send_responses_streaming(pending_responses)`
- Kuula `RequestInfoEvent` sündmust, et tuvastada, millal on vajalik inimese sisend
- Kuula `WorkflowOutputEvent` sündmust lõplike tulemiteni jõudmiseks


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Paris")], 
    should_respond=True
)

# Human-in-the-loop execution pattern
pending_responses: dict[str, str] | None = None
completed = False
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

while not completed:
    # First iteration uses run_stream with the request
    # Subsequent iterations use send_responses_streaming with collected human responses
    if pending_responses:
        print(f"\n📤 Sending human responses: {pending_responses}")
        stream = workflow.send_responses_streaming(pending_responses)
        pending_responses = None  # Clear immediately after sending
    else:
        print(f"\n🚀 Starting workflow with request: 'I want to book a hotel in Paris'")
        stream = workflow.run_stream(request_paris)
    
    # Collect all events from this iteration
    events = [event async for event in stream]
    
    # Process events
    requests: list[tuple[str, str]] = []  # (request_id, prompt)
    
    for event in events:
        # Check for human input requests
        if isinstance(event, RequestInfoEvent) and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            requests.append((event.request_id, event.data.prompt))
        
        # Check for workflow outputs
        elif isinstance(event, WorkflowOutputEvent):
            workflow_output = str(event.data)
            completed = True
            print(f"\n✅ Workflow completed with output!")
    
    # If we have human requests, prompt the user
    if requests and not completed:
        responses: dict[str, str] = {}
        for req_id, prompt in requests:
            print(f"\n{'='*60}")
            print(f"💬 QUESTION FOR YOU:")
            print(f"   {prompt}")
            print(f"{'='*60}")
            
            # Get user input (in notebook, this will pause execution)
            answer = input("👉 Enter 'yes' or 'no': ").strip().lower()
            
            print(f"\n📝 You answered: {answer}")
            responses[req_id] = answer
        
        pending_responses = responses

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## Samm 11: Käivita Testjuhtum 2 - Linn SAADAVUSEGA (Stockholm – ei vaja inimsekkumist)

See test näitab **otsest rada**, kui toad on saadaval:

1. Päringu hotell Stockholmis
2. availability_agent kontrollib → Toa saadaval ✅
3. booking_agent soovitab broneeringut
4. display_result kuvab kinnituse
5. **Inimesepanus pole vajalik!**

Töövoog jätab inimsekkumise tee täielikult vahele, kui toad on saadaval.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Stockholm")], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## Peamised järeldused ja parimad praktikad inimteguriga süsteemides

### ✅ Mida olete õppinud:

#### 1. **RequestInfoExecutor muster**
Microsoft Agent Frameworki inimteguriga muster kasutab kolme põhilist komponenti:
- `RequestInfoExecutor` - Peatab töövoo ja tekitab sündmusi
- `RequestInfoMessage` - Tüübistatud päringu andmete baasklass (pärige sellest!)
- `RequestResponse` - Seob inimeste vastused algsetega

**Oluline mõistmine:**
- `RequestInfoExecutor` ei kogu sisendit ise – ta ainult peatab töövoo
- Teie rakenduse kood peab kuulama `RequestInfoEvent` ja koguma sisendi
- Peate välja kutsuma `send_responses_streaming()` sõnastikuga, mis seob `request_id` kasutaja vastusega

#### 2. **Voogedastuse täitmise muster**
```python
# Esimene iteratsioon
stream = workflow.run_stream(initial_request)

# Järgmised iteratsioonid (pärast inimeste sisendit)
stream = workflow.send_responses_streaming(pending_responses)

# Töötle alati sündmusi
events = [event async for event in stream]
```

#### 3. **Sündmustepõhine arhitektuur**
Kuulake konkreetseid sündmusi töövoo juhtimiseks:
- `RequestInfoEvent` - Inimese sisend on vajalik (töövoog peatatud)
- `WorkflowOutputEvent` - Lõplik tulemus on saadaval (töövoog lõpetatud)
- `WorkflowStatusEvent` - Oleku muutused (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS jne)

#### 4. **Kohandatud täitjad koos @handleriga**
`DecisionManager` näitab, kuidas luua täitjaid, kes:
- Kasutavad `@handler` dekoratsiooni meetodite muutmiseks töövoo sammudeks
- Võtavad vastu tüübistatult sõnumeid (nt `RequestResponse[HumanFeedbackRequest, str]`)
- Suunavad töövoogu, saates sõnumeid teistele täitjatele
- Juurdepääs kontekstile läbi `WorkflowContext`

#### 5. **Tingimuslik marsruutimine inimeste otsuste põhjal**
Võite luua tingimusfunktsioone, mis hindavad inimeste vastuseid:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 Tõelised kasutusjuhtumid:

1. **Heakskiitmise töövood**
   - Saage juhi heakskiit enne kuludokumentide töötlemist
   - Nõuda inimkontrolli enne automatiseeritud meilide saatmist
   - Kinnitage kõrge väärtusega tehingud enne täitmist

2. **Sisu modereerimine**
   - Märgistage kahtlane sisu inimkontrolliks
   - Paluge moderaatoritel langetada lõplik otsus servajuhtumitel
   - Suunake inimesed sekkuma, kui tehisintellekti usaldus on madal

3. **Klienditeenindus**
   - Lase AI-l automaatselt käsitleda rutiinküsimusi
   - Suunake keerukad küsimused inimagendile
   - Küsi kliendilt, kas ta soovib rääkida inimesega

4. **Andmete töötlemine**
   - Paluge inimestel lahendada ebamäärased andmesisestused
   - Kinnitage AI tõlgendused ebamäärastest dokumentidest
   - Lase kasutajatel valida mitme kehtiva tõlgenduse vahel

5. **Turvameetmeid nõudvad süsteemid**
   - Nõua inimkinnitust enne pöördumatuid toiminguid
   - Saage heakskiit enne tundlike andmete kasutamist
   - Kinnitage otsused reguleeritud tööstusharudes (tervishoid, finants)

6. **Interaktiivsed agendid**
   - Ehitage vestlusbotid, mis esitavad täpsustavaid küsimusi
   - Looge viisardid, mis juhendavad kasutajaid keerulistes protsessides
   - Kujundage agendid, kes teevad koostööd inimestega samm-sammult

### 🔄 Võrdlus: Inimteguriga vs ilma

| Funktsioon | Tingimuslik töövoog | Inimteguriga töövoog |
|---------|---------------------|---------------------------|
| **Täitmine** | Üksik `workflow.run()` | Tsükkel `run_stream()` + `send_responses_streaming()` |
| **Kasutajasisend** | Puudub (täielikult automatiseeritud) | Interaktiivsed küsimused `input()` või UI kaudu |
| **Komponendid** | Agendid + Täitjad | + RequestInfoExecutor + DecisionManager |
| **Sündmused** | Ainult AgentExecutorResponse | RequestInfoEvent, WorkflowOutputEvent jpt |
| **Peatumine** | Ei peatu | Töövoog peatub RequestInfoExecutor juures |
| **Inimkontroll** | Inimene ei kontrolli | Inimesed teevad olulisi otsuseid |
| **Kasutusjuhtum** | Automatiseerimine | Koostöö ja järelevalve |

### 🚀 Täiustatud mustrid:

#### Mitmed inimotsuse punktid
Võite omada mitut `RequestInfoExecutor` sõlme samas töövoos:
```python
.add_edge(agent1, request_info_1)  # Esimene inimotsus
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # Teine inimotsus
.add_edge(decision_manager_2, final_agent)
```

#### Aegumise käsitlemine
Rakendage inimvastuste jaoks ajalimiite:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # Vaikevalik turvaliseks variandiks
```

#### Rikas UI integreerimine
`input()` asemel integreerige veebiliidese, Slacki, Teamsi jmt-ga:
```python
if isinstance(event, RequestInfoEvent):
    # Saada teavitus kasutaja eelistatud kanalisse
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### Tingimuslik inimteguriga sisend
Küsige inimelt sisendit ainult spetsiifilistes olukordades:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # Suuna inimesele ainult siis, kui usaldus on madal või väärtus kõrge
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ Parimad praktikad:

1. **Alati pärige RequestInfoMessage’ist**
   - Pakub tüübikindlust ja valideerimist
   - Võimaldab rikkalikku konteksti UI kuvamiseks
   - Selgitab iga päringu tüübi eesmärki

2. **Kasutage kirjeldavaid vihjeid**
   - Lisage kontekst, mida te küsitlete
   - Selgitage iga valiku tagajärgi
   - Hoidke küsimused lihtsad ja selged

3. **Käsitlege ootamatut sisendit**
   - Valideerige kasutaja vastuseid
   - Pakkuge vigase sisendi korral vaikeväärtusi
   - Andke selged veateated

4. **Jälgige request_id sidumist**
   - Kasutage seost request_id ja vastuste vahel
   - Ärge haldage olekut käsitsi

5. **Kujundage mittetõkkesiduvaks**
   - Ärge blokige lõime sisendi ootamise ajal
   - Kasutage kogu süsteemis asünkroonseid mustreid
   - Toetage samaaegseid töövoo eksemplare

### 📚 Seotud mõisted:

- **Agendisisene vahendkiht** - Agentkõnede pealtkuulamine (eelmine märkmik)
- **Töövoo oleku haldus** - Töövoo oleku säilitamine tööde vahel
- **Mitme agendi koostöö** - Ühendage inimtegur agentide tiimidega
- **Sündmustepõhised arhitektuurid** - Looge reaktiivseid süsteeme sündmuste abil

---

### 🎓 Õnnitleme!

Olete omandanud oskused Microsoft Agent Frameworki inimteguriga töövoogude haldamiseks! Nüüd oskate:
- ✅ Peatada töövood inimeste sisendi kogumiseks
- ✅ Kasutada RequestInfoExecutorit ja RequestInfoMessage’i
- ✅ Käsitleda voogedastusega täitmist sündmuste abil
- ✅ Luua kohandatud täitjaid koos @handleriga
- ✅ Suunata töövooge inimeste otsuste põhjal
- ✅ Ehita interaktiivseid tehisintellekti agente, kes teevad koostööd inimestega

**See on oluline muster usaldusväärsete ja kontrollitavate tehisintellekti süsteemide loomisel!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
